# DeBERTa-v3 Missing Seeds (Colab)
Runs seeds **13 and 7777** — the two seeds missing from the multi-seed paper results.
Seeds 42, 123, 2024 are already done locally; do NOT re-run them here.

batch_size=8, gradient_accumulation_steps=2 → effective batch 16. Fits in 22 GB (L4) and 40 GB (A100).

**Before running:** upload `repo.zip` then hit Runtime → Run all.

In [ ]:
# ── 1. Install dependencies ──────────────────────────────────────────────────
!pip install -q transformers==4.51.3 datasets accelerate scikit-learn pandas numpy

In [ ]:
# ── 2. Upload repo zip ───────────────────────────────────────────────────────
from google.colab import files
uploaded = files.upload()   # upload repo.zip

In [ ]:
# ── 3. Unzip and set working directory ───────────────────────────────────────
import zipfile, os

with zipfile.ZipFile('repo.zip', 'r') as z:
    z.extractall('/content/pids')

extracted = os.listdir('/content/pids')
if len(extracted) == 1 and os.path.isdir(f'/content/pids/{extracted[0]}'):
    os.chdir(f'/content/pids/{extracted[0]}')
else:
    os.chdir('/content/pids')

print('Working directory:', os.getcwd())
print('Contents:', os.listdir('.'))

In [ ]:
# ── 4. Verify GPU ────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── 5. Run DeBERTa seeds 42, 123, 2024 ───────────────────────────────────────
import sys, os
from pathlib import Path
sys.path.insert(0, '.')

from scripts.train_multi_seed import _append_row, _save_sweep, METRIC_COLS
from src.baselines.deberta_v3 import run_train

import torch, gc
gc.collect()
torch.cuda.empty_cache()

# batch_size=8 + grad_accum=2 → effective batch 16.
# Safe for both 22 GB (L4) and 40 GB (A100). All 3 seeds use identical settings.
BATCH_SIZE = 8
GRAD_ACCUM = 2

DEBERTA_RUNS = [("deberta", 42), ("deberta", 123), ("deberta", 2024)]

for i, (model_key, seed) in enumerate(DEBERTA_RUNS, 1):
    print(f'\n{"="*60}')
    print(f'[{i}/3]  deberta  seed={seed}')
    print(f'{"="*60}')
    gc.collect()
    torch.cuda.empty_cache()
    run_dir = Path('outputs/multi_seed_runs/deberta') / f'seed_{seed}'
    run_dir.mkdir(parents=True, exist_ok=True)
    metrics = run_train(
        num_epochs=3, batch_size=BATCH_SIZE, lr=2e-5,
        seed=seed, gradient_accumulation_steps=GRAD_ACCUM, out_dir=run_dir,
    )
    row = {'model_name': model_key, 'seed': seed}
    row.update(metrics)
    _append_row({col: row.get(col, None) for col in METRIC_COLS})
    _save_sweep(model_key, seed, run_dir)
    print(f'  -> Saved  IID_F1={metrics.get("IID_F1")}  hb_FPR_agg={metrics.get("hb_FPR_agg")}')

print('\nAll 3 seeds done!')

In [ ]:
# ── 6. Download results ───────────────────────────────────────────────────────
import zipfile, os
from google.colab import files

result_files = [
    'outputs/multi_seed_runs/all_runs.csv',
    'outputs/multi_seed_runs/sweep_deberta_seed42.csv',
    'outputs/multi_seed_runs/sweep_deberta_seed123.csv',
    'outputs/multi_seed_runs/sweep_deberta_seed2024.csv',
]

with zipfile.ZipFile('deberta_results.zip', 'w') as z:
    for f in result_files:
        if os.path.exists(f):
            z.write(f)
            print('Added:', f)
        else:
            print('MISSING:', f)

files.download('deberta_results.zip')

## After downloading

Back on your Mac, run these commands to merge the DeBERTa rows into your local master CSV:

```bash
# 1. Unzip the results
cd /Users/khalid/Projects/Prompt-Injection-Detector-System
unzip -o ~/Downloads/deberta_results.zip -d /tmp/deberta_results

# 2. Append only the deberta rows (skip header) to your local all_runs.csv
grep '^deberta' /tmp/deberta_results/outputs/multi_seed_runs/all_runs.csv \
  >> outputs/multi_seed_runs/all_runs.csv

# 3. Copy sweep files
cp /tmp/deberta_results/outputs/multi_seed_runs/sweep_deberta_seed*.csv \
   outputs/multi_seed_runs/

# 4. Generate final tables
python scripts/train_multi_seed.py --aggregate-only
```